In [1]:
import pandas as pd
import numpy as np

ruta=r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\SLA_Gold.parquet"
sla = pd.read_parquet(ruta)

# Asegurar que la columna de fecha sea datetime (evita comparaciones con strings)
sla["Fecha Finalizacion"] = pd.to_datetime(sla["Fecha Finalizacion"], errors="coerce")

decem = sla[sla["Fecha Finalizacion"] >= pd.Timestamp('2026-01-01')]

# Calcular medias solo para columnas numéricas y de tipo timedeltas (SLA)
num_mean = decem.select_dtypes(include=[np.number]).mean()
td_mean = decem.select_dtypes(include=['timedelta64[ns]']).mean()
mean_sla = pd.concat([num_mean, td_mean])

print("Mean SLA:")
print(mean_sla)

KeyError: 'Fecha Finalizacion'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Definir tu Top 5 basado en el análisis de Pareto
top_5_soluciones = [
    'CONECTOR DAÑADO',
    'CAMBIO DE VLAN',
    'CONFIGURACION DE EQUIPO',
    'FALLA GENERAL',
    'FALLA LOS'
]

# 2. Filtrar el DataFrame limpio (df_final) para quedarnos solo con el Top 5
df_pareto = df_final[df_final['Solucion Aplicada'].isin(top_5_soluciones)].copy()

# 3. Graficar los histogramas separados para ver la "limpieza" de las curvas
g = sns.FacetGrid(df_pareto, col="Solucion Aplicada", col_wrap=3, height=4, sharex=False, sharey=False)
g.map_dataframe(sns.histplot, x="SLA Resolucion Min", kde=True, bins=30, color='teal')

# Dar formato a los gráficos
g.set_titles(col_template="{col_name}")
g.set_axis_labels("Minutos de Resolución", "Cantidad de Tickets")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Supongamos que df_pareto es tu tabla ya filtrada con el Top 5 de soluciones
# 1. Definir el tamaño del "depósito" o bin (ej. cada 60 minutos = 1 hora)
ancho_bin = 60
max_minutos = int(df_pareto['SLA Resolucion Min'].max())
bins = range(0, max_minutos + ancho_bin, ancho_bin)

# 2. Crear las etiquetas (ej. "0-59", "60-119", etc.)
etiquetas = [f"{i} a {i+ancho_bin-1}" for i in bins[:-1]]

# 3. Asignar cada ticket a su rango correspondiente
df_pareto['Rango_Tiempo_Min'] = pd.cut(df_pareto['SLA Resolucion Min'], bins=bins, labels=etiquetas, right=False)

# 4. AGRUPAR: Esta es la magia que comprime los datos para Power BI
df_gold_distribucion = df_pareto.groupby(['Solucion Aplicada', 'Rango_Tiempo_Min']).size().reset_index(name='Cantidad_Tickets')

# Limpiar rangos donde no hubo ningún ticket para ahorrar aún más peso
df_gold_distribucion = df_gold_distribucion[df_gold_distribucion['Cantidad_Tickets'] > 0]
display(df_gold_distribucion)
# Exportar a Parquet
# df_gold_distribucion.to_parquet("Gold_Distribucion_SLA.parquet")

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Listado_Gold.parquet')
display(df)

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Recaudacion_Gold.parquet')
print(df.columns)
df = df.groupby(['Hora de Pago', 'Vendedor'], as_index=False).size()


display(df)

In [ ]:
import polars as pl
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Downloads\Ventas_Estatus_Gold-Sharepoint.parquet')

mask = (df['Fecha']>=('2026-02-01')) & (df['Fecha']<=('2026-02-28')) &(df['Vendedor'].str.contains("OFI", case=False, na=False))
df_feb=df[mask]
print(df_feb.columns)
df_feb = df_feb.drop_duplicates(subset=['N° Abonado', 'Documento','Fecha','Vendedor','Ciudad','Hora'])
display(df_feb.describe(include='all'))
display(df_feb)

In [ ]:
import pandas as pd
import polars as pl
from pathlib import Path
import duckdb

rutahome = Path.home()
ruta_base = rutahome / 'Documents' / 'A-DataStack' / '01-Proyectos' / '01-Data_PipelinesFibex' / '02_Data_Lake' / 'gold_data'

lf = duckdb.query


In [3]:
import pandas as pd 

df_ventas= pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Listado_Gold.parquet")
df_estatus= pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Estatus_Gold.parquet")
df_afluencia = pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Afluencia_Gold.parquet")
#display(df_estatus)
fecha = '2026-04-01'
mask_ofi_march = (df_ventas['Canal'] == 'OFICINA COMERCIAL') & (df_ventas['Fecha Contrato'] >= fecha)
mask_est_march = (df_estatus['Fecha'] >= fecha)
mask_ventas_march = (df_afluencia['Fecha']>=fecha) & (df_afluencia['Tipo de afluencia'] == 'Ventas')

df_ventas = df_ventas[mask_ofi_march]
df_estatus= df_estatus[mask_est_march]
df_afluencia = df_afluencia[mask_ventas_march]



display(df_ventas.groupby(['Estatus']).agg(Cantidad = ('Estatus','size')))
display(df_estatus.groupby(['Estatus']).agg(Cantidad = ('Estatus','size')))
display(df_afluencia.groupby(['Estatus']).agg(Cantidad = ('Estatus','size')))

,Cantidad
Estatus,
ACTIVO,294
OBSTRUCCION,3
POR IMPLEMENTACION,14
POR INSTALAR,55
POR VGT,1
SIN FACTIBILIDAD,3


,Cantidad
Estatus,
ACTIVO,298
ANULADO,5
OBSTRUCCION,3
POR IMPLEMENTACION,14
POR INSTALAR,47
POR VGT,1
RECHAZADO,2
REEMBOLSO ADM,3
REEMBOLSO PAGADOS,1


,Cantidad
Estatus,
ACTIVO,298
ANULADO,5
OBSTRUCCION,3
POR IMPLEMENTACION,14
POR INSTALAR,47
POR VGT,1
RECHAZADO,2
REEMBOLSO ADM,3
REEMBOLSO PAGADOS,1


In [2]:
import pandas as pd


df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Horas_Raw_Bronze.parquet')
df_recaudacion = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Recaudacion_Raw_Bronze.parquet')
display(df.columns)
display(df)
display(df_recaudacion.columns)
display(df_recaudacion)
df_merge = pd.merge(df_recaudacion, df, on='ID Pago', how='inner')
display(df_merge.head())

Index(['ID Contrato', 'ID Pago', 'ID pago', 'N° Abonado', 'Fecha', 'Cobrador',
       'Hora de Pago', 'Oficina Cobro', 'Forma de Pago', 'Source.Name',
       'Documento', 'Cliente', 'Nro Recibo', 'Total Pago', 'Total Pago Bs',
       'Tasa de cambio del día', 'Suscripción', 'Usuario de Anulacion',
       'Fecha_Modificacion_Archivo'],
      dtype='object')

,ID Contrato,ID Pago,ID pago,N° Abonado,Fecha,Cobrador,Hora de Pago,Oficina Cobro,Forma de Pago,Source.Name,Documento,Cliente,Nro Recibo,Total Pago,Total Pago Bs,Tasa de cambio del día,Suscripción,Usuario de Anulacion,Fecha_Modificacion_Archivo
0,'CONT5FAA8142C6951765','6939BE7F00D612779546',None,LCH25634,2025-12-10 00:00:00,None,1899-12-31 14:45:00,OFIC-BOCA DE UCHIRE,None,Data-HorasPago 10-12 al 20-1.xlsx,14428593,None,467757,None,None,None,None,None,2026-01-20 13:07:39.202745
1,'CONT3B49B198EB478425','689E4CD54918F7129118',689E4CD54918F7129118,LCH60981,2025-08-14 00:00:00,AURIMIR DEL VALLE GUEVARA OFIC CUMANA,1899-12-31 16:55:00,OFC COMERCIAL CUMANA,EFECTIVO - 30.06,Data-HorasPago 01-08 al 27-11.xlsx,None,None,None,None,None,None,None,None,NaT
2,'CONTA941BF1B27193586','68C180DC31C121438706',68C180DC31C121438706,GU5966,2025-09-10 00:00:00,GERONIMO A ZAMBRANO ASESOR OFICINA GUARICO,1899-12-31 09:47:00,OFC - VALLE LA PASCUA,TARJETA DE DEBITO - 15.00,Data-HorasPago 01-08 al 27-11.xlsx,None,None,None,None,None,None,None,None,2025-12-02 10:03:02.810474
3,'CONTA280781184E66603','68A61FB99DEAE2115681',68A61FB99DEAE2115681,CHV1618,2025-08-20 00:00:00,YVANNA GABRIELA MARCHAN ATC SERTEMAC,1899-12-31 15:24:00,OFIC-SERTEMAC-CHIVACOA,TARJETA DE DEBITO - 12.05,Data-HorasPago 01-08 al 27-11.xlsx,None,None,None,None,None,None,None,None,2025-12-02 10:03:02.810474
4,'CONT3391B02EA0704648','6963B7C0605D53178327',None,V62523,2026-01-11 00:00:00,None,1899-12-31 10:46:00,OFI-VIRTUAL,None,Data-HorasPago 10-12 al 20-1.xlsx,7123954,None,342209,None,None,None,None,None,2026-01-20 13:07:39.202745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5426641,'CONT8A1D744C7E557557','68C47A55E19501675739',68C47A55E19501675739,LV4256,2025-09-12 00:00:00,ELIANNIS DANIELA SEVILLA AGNTE AUTORIZADO ARKN...,1899-12-31 15:55:00,OFIC ARKNET JAD SAN CARLOS CENTRO,TARJETA DE DEBITO - 19.99,Data-HorasPago 01-08 al 27-11.xlsx,None,None,None,None,None,None,None,None,NaT
5426642,'CONTD1C31837F1997734','686819CD54DE67749551',None,LCH49478,2025-07-04 00:00:00,None,1899-12-31 14:16:00,OFI-BARCELONA,None,Data-HorasPago 1-7 al 31-7.xlsx,18567657,None,346908,None,None,None,None,None,2026-01-20 15:06:28.142336
5426643,'CONT94E3E8F347288679','697229E320EA33480803',None,POR0013,2026-01-22 00:00:00,None,1899-12-31 09:48:00,OFI-BOCONOITO,None,Data-HorasPago 22-1 al 22-1.xlsx,13738088,None,003526,None,None,None,None,None,NaT
5426644,'CONT44A90F4545539756','6962732439A853028308',None,V3933,2026-01-10 00:00:00,None,1899-12-31 11:42:00,OFI-METROPOLIS,None,Data-HorasPago 30-1 al 2-2.xlsx,6266682,None,094281,None,None,None,40,None,NaT


Index(['ID Contrato', 'ID Pago', 'N° Abonado', 'Nro Recibo', 'Fecha',
       'Total Pago', 'Total Pago Bs', 'ID pago', 'Estatus Pago',
       'Forma de Pago', 'Monto Forma Pago', 'Tasa de cambio del día', 'Banco',
       'Referencia', 'Cobrador', 'Nombre Caja', 'Oficina Cobro',
       'Fecha Contrato', 'Estatus', 'Suscripción', 'Etiqueta',
       'Grupo Afinidad', 'Nombre Franquicia', 'Ciudad', 'Source.Name',
       'Fecha_Modificacion_Archivo'],
      dtype='object')

,ID Contrato,ID Pago,N° Abonado,Nro Recibo,Fecha,Total Pago,Total Pago Bs,ID pago,Estatus Pago,Forma de Pago,...,Oficina Cobro,Fecha Contrato,Estatus,Suscripción,Etiqueta,Grupo Afinidad,Nombre Franquicia,Ciudad,Source.Name,Fecha_Modificacion_Archivo
0,'CONT8E932386B3200796','6957D6473D0675791646',V71522,249114,2026-01-02,35,10547.98,6957D6473D0675791646,PAGADO,TARJETA DE DEBITO,...,OFIC. TORRE FIBEX VIÑEDO,2024-07-10 00:00:00,ACTIVO,35,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-1-2026 al 14-1-2026.xlsx,NaT
1,'CONT543CB698DDA71852','695820C9642025193692',V4347,249505,2026-01-02,20,6027.42,695820C9642025193692,PAGADO,TARJETA DE DEBITO,...,OFIC. TORRE FIBEX VIÑEDO,2022-04-11 00:00:00,ACTIVO,20,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-1-2026 al 14-1-2026.xlsx,NaT
2,'CONT96D22C1D2D962215','6958210405DFE8263563',H5188,249507,2026-01-02,40,12054.84,6958210405DFE8263563,PAGADO,TARJETA DE DEBITO,...,OFIC. TORRE FIBEX VIÑEDO,2021-05-08 00:00:00,ACTIVO,40,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-1-2026 al 14-1-2026.xlsx,NaT
3,'CONT650281039D721511','6957C92C4857D9211530',LA2046,082770,2026-01-02,6.64,2000,6957C92C4857D9211530,PAGADO,EFECTIVO,...,OFC LAS ADJUNTAS,2023-11-27 00:00:00,ACTIVO,30,None,FIBEX EXPRESS,FIBEX LAS ADJUNTAS,CARACAS,Data - Recaudacion 1-1-2026 al 14-1-2026.xlsx,NaT
4,'CONT4E38E96AF2C59678','695812A7935F41748900',H9863,122687,2026-01-02,20,6027.42,695812A7935F41748900,PAGADO,TARJETA DE DEBITO,...,OFI TINAQUILLO,2021-09-24 00:00:00,ACTIVO,20,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-1-2026 al 14-1-2026.xlsx,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235855,'CONT373ACE29CD133923','69D8E9B27F7E71782019',GU25674,363088,2026-04-10,40,18957,69D8E9B27F7E71782019,PAGADO,EFECTIVO,...,OFIC-ZARAZA,2024-11-15 00:00:00,ACTIVO,40,ANALISTA 7,PYMES,FIBEX GUARICO,ZARAZA,Data - Recaudacion 9-4-2026 al 10-4-2026.xlsx,2026-04-10 08:20:20.884176
2235856,'CONT81C951CE38A42321','69D8EA123C91F6529118',GU4720,363116,2026-04-10,46.4,21989,69D8EA123C91F6529118,PAGADO,EFECTIVO,...,OFIC-ZARAZA,2023-12-19 00:00:00,ACTIVO,46.4,ANALISTA 7,PYMES,FIBEX GUARICO,ZARAZA,Data - Recaudacion 9-4-2026 al 10-4-2026.xlsx,2026-04-10 08:20:20.884176
2235857,'CONT7C75C4581FB45934','69D8EA33827ED7080592',GU4445,363131,2026-04-10,116,54974,69D8EA33827ED7080592,PAGADO,EFECTIVO,...,OFIC-ZARAZA,2023-12-15 00:00:00,ACTIVO,116,ANALISTA 7,PYMES,FIBEX GUARICO,ZARAZA,Data - Recaudacion 9-4-2026 al 10-4-2026.xlsx,2026-04-10 08:20:20.884176
2235858,'CONTA4C6290E56214978','69D8EA490CF7E1905193',LCH72642,363147,2026-04-10,40,18956,69D8EA490CF7E1905193,PAGADO,EFECTIVO,...,OFIC-ZARAZA,2025-08-19 00:00:00,ACTIVO,40,ANALISTA 6,PYMES,FIBEX ANZOATEGUI,PUERTO LA CRUZ,Data - Recaudacion 9-4-2026 al 10-4-2026.xlsx,2026-04-10 08:20:20.884176


,ID Contrato_x,ID Pago,N° Abonado_x,Nro Recibo_x,Fecha_x,Total Pago_x,Total Pago Bs_x,ID pago_x,Estatus Pago,Forma de Pago_x,...,Source.Name_y,Documento,Cliente,Nro Recibo_y,Total Pago_y,Total Pago Bs_y,Tasa de cambio del día_y,Suscripción_y,Usuario de Anulacion,Fecha_Modificacion_Archivo_y
0,'CONT8E932386B3200796','6957D6473D0675791646',V71522,249114,2026-01-02,35,10547.98,6957D6473D0675791646,PAGADO,TARJETA DE DEBITO,...,Data-HorasPago 30-1 al 2-2.xlsx,23411935,None,249114,None,None,None,35,None,2026-02-02 08:53:00.515741
1,'CONT8E932386B3200796','6957D6473D0675791646',V71522,249114,2026-01-02,35,10547.98,6957D6473D0675791646,PAGADO,TARJETA DE DEBITO,...,Data-HorasPago 30-1 al 2-2.xlsx,23411935,None,249114,None,None,None,35,None,NaT
2,'CONT8E932386B3200796','6957D6473D0675791646',V71522,249114,2026-01-02,35,10547.98,6957D6473D0675791646,PAGADO,TARJETA DE DEBITO,...,Data-HorasPago 10-12 al 20-1.xlsx,23411935,None,249114,None,None,None,None,None,NaT
3,'CONT8E932386B3200796','6957D6473D0675791646',V71522,249114,2026-01-02,35,10547.98,6957D6473D0675791646,PAGADO,TARJETA DE DEBITO,...,Data-HorasPago 10-12 al 20-1.xlsx,23411935,None,249114,None,None,None,None,None,2026-01-20 13:07:39.202745
4,'CONT543CB698DDA71852','695820C9642025193692',V4347,249505,2026-01-02,20,6027.42,695820C9642025193692,PAGADO,TARJETA DE DEBITO,...,Data-HorasPago 10-12 al 20-1.xlsx,15000698,None,249505,None,None,None,None,None,NaT


In [2]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Recaudacion_Gold.parquet')
display(df)

,ID Contrato,N° Abonado,Fecha,Total Pago,Forma de Pago,Banco,Oficina,Fecha Contrato,Estatus,Suscripción,Grupo Afinidad,Nombre Franquicia,Ciudad,Vendedor,Tipo de afluencia,Clasificacion,Hora de Pago
0,'CONT21FE5D5030705988',LCH58251,2025-07-01,16.64,TARJETA DE DEBITO,PROVINCIAL,OFC COMERCIAL CUMANA,2025-05-12 00:00:00,ACTIVO,30,HOGAR,FIBEX ANZOATEGUI,CUMANA,JORGE PAREJO OFIC CUMANA,RECAUDACIÓN,OFICINAS PROPIAS,07:00
1,'CONT82B7255301054845',GU17461,2025-07-01,30.00,TARJETA DE DEBITO,100% BANCO TH,OFC - VALLE LA PASCUA,2024-07-01 00:00:00,ACTIVO,30,HOGAR,FIBEX GUARICO,VALLE DE LA PASCUA,GERONIMO A ZAMBRANO ASESOR OFICINA GUARICO,RECAUDACIÓN,ALIADOS Y DESARROLLO,08:00
2,'CONT3CA2F6D7A1834761',PTO10344,2025-07-01,30.00,TARJETA DE DEBITO,PROVINCIAL,OFI-PTO CABELLO,2024-11-19 00:00:00,ACTIVO,30,HOGAR,FIBEX PTO CABELLO,PUERTO CABELLO,CARILYN MELENDEZ OFIC PUERTO CABELLO,RECAUDACIÓN,OFICINAS PROPIAS,08:00
3,'CONT9E17A54A36F85147',LV2565,2025-07-01,20.00,TARJETA DE DEBITO,BANCO NACIONAL CREDITO,OFC-INTERCOM SAN CARLOS LAS VEGAS,2025-02-01 00:00:00,ACTIVO,25,HOGAR,FIBEX LAS VEGAS,LAS VEGAS,INES MARIA LABARCA BELTRAN AGENTE AUTORIZADO I...,RECAUDACIÓN,ALIADOS Y DESARROLLO,08:00
4,'CONT0F889188DCA55438',LV3444,2025-07-01,10.00,TARJETA DE DEBITO,BANCO NACIONAL CREDITO,OFC-INTERCOM SAN CARLOS LAS VEGAS,2025-04-28 00:00:00,CORTADO,25,HOGAR,FIBEX LAS VEGAS,LAS VEGAS,INES MARIA LABARCA BELTRAN AGENTE AUTORIZADO I...,RECAUDACIÓN,ALIADOS Y DESARROLLO,08:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1537689,'CONT6625705F0D043154',FM1739,2026-03-06,35.00,TARJETA DE DEBITO,PROVINCIAL,OFC COBRO CENTRO,2023-11-28 00:00:00,ACTIVO,35,HOGAR,FIBEX MATURIN,MATURIN,ELOINA ARTEAGA,RECAUDACIÓN,ALIADOS Y DESARROLLO,13:00
1537690,'CONTC07586C5D9929991',V9476,2026-03-06,39.99,TARJETA DE DEBITO,100% BANCO,OFI-METROPOLIS,2022-07-02 00:00:00,ACTIVO,42,HOGAR,FIBEX VALENCIA,VALENCIA,MAYERLIN GARCIA WALTER OFIC METROPOLIS VALENCIA,RECAUDACIÓN,OFICINAS PROPIAS,13:00
1537691,'CONTB747062EC6A45817',C2933,2026-03-06,40.00,EFECTIVO DOLLAR,,OFI-PARAISO,2023-01-05 00:00:00,ACTIVO,42,HOGAR,FIBEX CARACAS,CARACAS,KARIANNYS RODRIGUEZ OFIC EL PARAISO CARACAS,RECAUDACIÓN,OFICINAS PROPIAS,13:00
1537692,'CONTA384F1D4C4149627',V122291,2026-03-06,25.00,TARJETA DE DEBITO,PROVINCIAL,OFC - GUACARA,2025-08-18 00:00:00,ACTIVO,25,HOGAR,FIBEX VALENCIA,VALENCIA,YELIUSKAR MELENDEZ AGENTE ESCALA 23 GUACARA,RECAUDACIÓN,ALIADOS Y DESARROLLO,13:00
